# Inspection classification: coefficient interpretation

This notebook loads the saved logistic-regression pipeline and calculates the odds ratio for every transformed feature:

$$\text{odds ratio} = e^{\beta}$$

An odds ratio above 1 is associated with higher odds of a VAI/OAI classification; an odds ratio below 1 is associated with lower odds. These are associations, not causal effects.

## 1. Locate the project and load dependencies

Run this notebook from anywhere inside the project. If necessary, set the `QUALIFYZE_PROJECT_ROOT` environment variable to the repository root.

In [ ]:
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


def find_project_root(start: Path) -> Path:
    override = os.environ.get("QUALIFYZE_PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if not (root / "pyproject.toml").is_file():
            raise FileNotFoundError(
                "QUALIFYZE_PROJECT_ROOT does not contain pyproject.toml: "
                f"{root}"
            )
        return root

    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Run the notebook from inside "
        "the repository or set QUALIFYZE_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## 2. Resolve and load the configured model artifact

In [ ]:
from qualifyze.config import Settings

settings = Settings()  # type: ignore
model_config = settings.modeling.inspection_classification

artifact_root = Path(model_config.artifact_root).expanduser()
if not artifact_root.is_absolute():
    artifact_root = PROJECT_ROOT / artifact_root

artifact_path = (
    artifact_root
    / model_config.model_version
    / "model.joblib"
)

if not artifact_path.is_file():
    raise FileNotFoundError(
        f"Model artifact not found: {artifact_path}. "
        "Train the model first or check modeling.inspection_classification "
        "in config.yml."
    )

artifact = joblib.load(artifact_path)
pipeline = artifact["pipeline"]
threshold = float(artifact.get("threshold", 0.5))

artifact_summary = pd.Series(
    {
        "artifact_path": str(artifact_path),
        "model_name": artifact.get("model_name"),
        "model_version": artifact.get("model_version"),
        "dataset_version": artifact.get("dataset_version"),
        "trained_at": artifact.get("trained_at"),
        "test_start_date": artifact.get("test_start_date"),
        "decision_threshold": threshold,
    },
    name="value",
)
display(artifact_summary.to_frame())

## 3. Check the fitted estimator

Interpret coefficients only after confirming that the estimator converged. The coefficient vector for a binary sklearn logistic regression corresponds to `classes_[1]`.

In [ ]:
required_steps = {"feature_processing", "classifier"}
missing_steps = required_steps - set(pipeline.named_steps)
if missing_steps:
    raise ValueError(
        f"The saved pipeline is missing steps: {sorted(missing_steps)}"
    )

processor = pipeline.named_steps["feature_processing"]
classifier = pipeline.named_steps["classifier"]

if not hasattr(classifier, "coef_"):
    raise TypeError(
        "The saved classifier does not expose logistic-regression coefficients."
    )

iterations = int(np.max(classifier.n_iter_))
maximum_iterations = int(classifier.max_iter)
converged = iterations < maximum_iterations
positive_class = classifier.classes_[1]

estimator_summary = pd.Series(
    {
        "estimator": type(classifier).__name__,
        "solver": classifier.solver,
        "classes": classifier.classes_.tolist(),
        "coefficient_target_class": positive_class,
        "iterations": iterations,
        "maximum_iterations": maximum_iterations,
        "converged": converged,
    },
    name="value",
)
display(estimator_summary.to_frame())

if not converged:
    print(
        "WARNING: The estimator reached max_iter. Treat coefficient "
        "interpretations as provisional and retrain a converged model."
    )

## 4. Extract coefficients and calculate odds ratios

The table includes the raw log-odds coefficient, its exponential, and the corresponding percentage change in odds.

In [ ]:
transformed_feature_names = processor.get_feature_names_out()
coefficients = np.asarray(classifier.coef_[0], dtype=float)

if len(transformed_feature_names) != len(coefficients):
    raise ValueError(
        "The number of transformed feature names does not match the "
        "number of model coefficients."
    )


def clean_feature_name(name: str) -> str:
    cleaned = name
    for prefix in ("continuous__", "binary__", "categorical__"):
        if cleaned.startswith(prefix):
            cleaned = cleaned.removeprefix(prefix)
            break

    if cleaned.startswith("state_"):
        return f"State: {cleaned.removeprefix('state_')}"

    return cleaned.replace("_", " ").title()


coefficient_table = pd.DataFrame(
    {
        "transformed_feature": transformed_feature_names,
        "feature": [
            clean_feature_name(name)
            for name in transformed_feature_names
        ],
        "coefficient": coefficients,
        "odds_ratio": np.exp(coefficients),
    }
)

coefficient_table["odds_change_percent"] = (
    coefficient_table["odds_ratio"] - 1.0
) * 100.0

coefficient_table["direction"] = np.select(
    [
        coefficient_table["coefficient"] > 0,
        coefficient_table["coefficient"] < 0,
    ],
    [
        "Higher adverse odds",
        "Lower adverse odds",
    ],
    default="No estimated change",
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient",
    ascending=False,
).reset_index(drop=True)

display(
    coefficient_table.drop(
        columns="absolute_coefficient"
    ).style.format(
        {
            "coefficient": "{:.4f}",
            "odds_ratio": "{:.3f}",
            "odds_change_percent": "{:+.1f}%",
        }
    )
)

## 5. Largest positive and negative associations

In [ ]:
display_columns = [
    "feature",
    "coefficient",
    "odds_ratio",
    "odds_change_percent",
]

positive_associations = coefficient_table.nlargest(
    15,
    "coefficient",
)[display_columns]

negative_associations = coefficient_table.nsmallest(
    15,
    "coefficient",
)[display_columns]

print("Features associated with higher VAI/OAI odds")
display(
    positive_associations.style.format(
        {
            "coefficient": "{:.4f}",
            "odds_ratio": "{:.3f}",
            "odds_change_percent": "{:+.1f}%",
        }
    )
)

print("Features associated with lower VAI/OAI odds")
display(
    negative_associations.style.format(
        {
            "coefficient": "{:.4f}",
            "odds_ratio": "{:.3f}",
            "odds_change_percent": "{:+.1f}%",
        }
    )
)

## 6. Odds-ratio chart

The chart uses a logarithmic x-axis so that values below and above 1 are visually comparable.

In [ ]:
top_n = 20
plot_data = (
    coefficient_table.nlargest(
        top_n,
        "absolute_coefficient",
    )
    .sort_values("odds_ratio")
)

colors = np.where(
    plot_data["odds_ratio"] >= 1,
    "#D55E00",
    "#0072B2",
)

fig, ax = plt.subplots(figsize=(11, 9))
ax.barh(
    plot_data["feature"],
    plot_data["odds_ratio"],
    color=colors,
    alpha=0.85,
)
ax.axvline(1.0, color="#333333", linewidth=1.2, linestyle="--")
ax.set_xscale("log")
ax.set_xlabel("Odds ratio, logarithmic scale")
ax.set_ylabel("")
ax.set_title(
    f"Largest logistic-regression associations with class {positive_class}"
)
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
plot_data = (
    state_odds_ratios
    .dropna(
        subset=[
            "state",
            "odds_change_percent",
        ]
    )
    .sort_values("odds_change_percent")
    .reset_index(drop=True)
)


if plot_data.empty:
    raise ValueError(
        "No state odds ratios are available to plot."
    )


# Baseline: dark grey
# Higher odds: orange
# Lower odds: blue
state_colors = np.where(
    plot_data["is_baseline"],
    "#333333",
    np.where(
        plot_data["odds_change_percent"] >= 0,
        "#E34A3A",
        "#618FFC",
    ),
)


# Wide, relatively short figure suitable for a slide.
fig, ax = plt.subplots(
    figsize=(16, 6),
    constrained_layout=True,
)


# ax.bar returns one matplotlib bar object per state.
bars = ax.bar(
    x=plot_data["state"],
    height=plot_data["odds_change_percent"],
    color=state_colors,
    alpha=0.85,
    width=0.8,
    zorder=2,
)


# Zero means equal odds to the omitted baseline state.
ax.axhline(
    y=0,
    color="#333333",
    linewidth=1.2,
    linestyle="--",
    zorder=3,
)


# Values already use percentage units:
# 25 means 25%, not 0.25.
ax.yaxis.set_major_formatter(
    PercentFormatter(
        xmax=100,
        decimals=0,
    )
)


ax.set_xlabel("State")

ax.set_ylabel(
    f"Change in odds relative to {baseline_state}"
)

ax.set_title(
    "Estimated change in VAI/OAI odds by state"
)


# Rotate the state names to fit the slide.
ax.tick_params(
    axis="x",
    labelrotation=80,
    labelsize=8,
)


# Place grid lines behind the bars.
ax.set_axisbelow(True)

ax.grid(
    axis="y",
    alpha=0.2,
)


# Expand both ends of the y-axis so that percentage
# annotations remain inside the figure.
values = plot_data[
    "odds_change_percent"
].to_numpy()

data_min = min(
    0.0,
    float(values.min()),
)

data_max = max(
    0.0,
    float(values.max()),
)

data_range = max(
    data_max - data_min,
    1.0,
)

padding = data_range * 0.18

ax.set_ylim(
    data_min - padding,
    data_max + padding,
)


# Add the percentage at the end of each bar.
for bar, value in zip(
    bars,
    values,
):
    if value >= 0:
        vertical_alignment = "bottom"
        label_offset = 3
    else:
        vertical_alignment = "top"
        label_offset = -3

    ax.annotate(
        text=f"{value:+.0f}%",
        xy=(
            bar.get_x()
            + bar.get_width() / 2,
            value,
        ),
        xytext=(
            0,
            label_offset,
        ),
        textcoords="offset points",
        ha="center",
        va=vertical_alignment,
        rotation=90,
        fontsize=9,
        clip_on=True,
    )


plt.show()

In [ ]:
product_prefix = "binary__has_prior_product_"


# Extract the product-indicator coefficients.
product_odds_ratios = coefficient_table.loc[
    coefficient_table[
        "transformed_feature"
    ].str.startswith(product_prefix),
    [
        "transformed_feature",
        "coefficient",
        "odds_ratio",
        "odds_change_percent",
    ],
].copy()


# Create readable product names.
product_odds_ratios["product"] = (
    product_odds_ratios["transformed_feature"]
    .str.removeprefix(product_prefix)
    .str.replace("_", " ", regex=False)
    .str.title()
)


plot_data = (
    product_odds_ratios
    .dropna(
        subset=[
            "product",
            "odds_change_percent",
        ]
    )
    .sort_values("odds_change_percent")
    .reset_index(drop=True)
)


if plot_data.empty:
    raise ValueError(
        "No has_prior_product features were found. "
        f"Expected transformed features beginning with "
        f"{product_prefix!r}."
    )


# Higher odds: orange
# Lower odds: blue
product_colors = np.where(
    plot_data["odds_change_percent"] >= 0,
    "#E34A3A",
    "#618FFC",
)


fig, ax = plt.subplots(
    figsize=(12, 6),
    constrained_layout=True,
)


bars = ax.bar(
    x=plot_data["product"],
    height=plot_data["odds_change_percent"],
    color=product_colors,
    alpha=0.85,
    width=0.7,
    zorder=2,
)


# The baseline for each product feature is:
# has_prior_product_* = 0.
ax.axhline(
    y=0,
    color="#252525",
    linewidth=1.2,
    linestyle="--",
    label="No prior product history (baseline)",
    zorder=3,
)


ax.yaxis.set_major_formatter(
    PercentFormatter(
        xmax=100,
        decimals=0,
    )
)


ax.set_xlabel("Prior product type")

ax.set_ylabel(
    "Change in estimated VAI/OAI odds"
)

ax.set_title(
    "Estimated change in VAI/OAI odds by prior product type"
)


ax.tick_params(
    axis="x",
    labelrotation=35,
    labelsize=9,
)


ax.set_axisbelow(True)

ax.grid(
    axis="y",
    alpha=0.2,
)


# Expand the y-axis so percentage labels remain visible.
values = plot_data[
    "odds_change_percent"
].to_numpy()

data_min = min(
    0.0,
    float(values.min()),
)

data_max = max(
    0.0,
    float(values.max()),
)

data_range = max(
    data_max - data_min,
    1.0,
)

padding = data_range * 0.18

ax.set_ylim(
    data_min - padding,
    data_max + padding,
)


# Add percentage labels.
for bar, value in zip(
    bars,
    values,
):
    if value >= 0:
        vertical_alignment = "bottom"
        label_offset = 3
    else:
        vertical_alignment = "top"
        label_offset = -3

    ax.annotate(
        text=f"{value:+.1f}%",
        xy=(
            bar.get_x()
            + bar.get_width() / 2,
            value,
        ),
        xytext=(
            0,
            label_offset,
        ),
        textcoords="offset points",
        ha="center",
        va=vertical_alignment,
        fontsize=10,
        clip_on=True,
    )


ax.legend(
    loc="best",
    frameon=False,
)


plt.show()

## 7. Identify categorical reference levels

With `OneHotEncoder(drop='first')`, each state coefficient is interpreted relative to the omitted reference state.

In [ ]:
reference_rows = []

try:
    categorical_pipeline = processor.named_transformers_["categorical"]
    encoder = categorical_pipeline.named_steps["encoder"]
    categorical_columns = next(
        columns
        for name, _, columns in processor.transformers_
        if name == "categorical"
    )

    drop_indices = getattr(encoder, "drop_idx_", None)
    for index, (column, categories) in enumerate(
        zip(categorical_columns, encoder.categories_, strict=True)
    ):
        dropped = None
        if drop_indices is not None and drop_indices[index] is not None:
            dropped = categories[int(drop_indices[index])]

        reference_rows.append(
            {
                "categorical_feature": column,
                "reference_level": dropped,
                "number_of_observed_levels": len(categories),
            }
        )
except (KeyError, AttributeError, StopIteration) as exc:
    print(f"Could not resolve categorical reference levels: {exc}")

reference_levels = pd.DataFrame(reference_rows)
display(reference_levels)

## 8. Review saved test metrics

In [ ]:
saved_metrics = artifact.get("metrics", {})
if saved_metrics:
    display(
        pd.Series(saved_metrics, name="value")
        .rename_axis("metric")
        .to_frame()
    )
else:
    print("No metrics were stored in the artifact.")

In [ ]:
# 9. Continuous variables

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import PercentFormatter


QUALIFYZE_COLORS = {
    "higher_odds": "#E34A3A",
    "lower_odds": "#618FFC",
    "baseline": "#33374D",
    "grid": "#DCE6FF",
    "background": "#FFFFFF",
    "text": "#33374D",
}


CONTINUOUS_FEATURE_LABELS = {
    "prior_inspection_count":
        "Prior inspections",
    "days_since_previous_inspection":
        "Days since previous inspection",
    "historical_product_type_count":
        "Historical product types",
    "prior_citation_count":
        "Prior citations",
    "previous_inspection_citation_count":
        "Previous inspection citations",
    "prior_citations_per_inspection":
        "Prior citations per inspection",
    "repeated_cfr_count":
        "Repeated CFR provisions",
    "prior_published_483_count_per_inspection":
        "Published 483s per inspection",
    "prior_warning_letter_count_per_inspection":
        "Warning letters per inspection",
    "prior_recall_event_count_per_inspection":
        "Recall events per inspection",
}


def identify_continuous_feature(
    transformed_feature: str,
) -> str | None:
    for feature in CONTINUOUS_FEATURE_LABELS:
        if (
            transformed_feature == feature
            or transformed_feature.endswith(
                f"__{feature}"
            )
        ):
            return feature

    return None


continuous_odds_ratios = coefficient_table.copy()

continuous_odds_ratios["raw_feature"] = (
    continuous_odds_ratios[
        "transformed_feature"
    ].apply(identify_continuous_feature)
)

continuous_odds_ratios = (
    continuous_odds_ratios.loc[
        continuous_odds_ratios[
            "raw_feature"
        ].notna()
    ].copy()
)


if continuous_odds_ratios.empty:
    raise ValueError(
        "No continuous model features were found."
    )


continuous_odds_ratios["variable"] = (
    continuous_odds_ratios["raw_feature"].map(
        CONTINUOUS_FEATURE_LABELS
    )
)


# Because these variables were not standardized,
# exp(coefficient) is the odds ratio for a one-unit increase
# in the original variable.
continuous_odds_ratios["odds_ratio"] = np.exp(
    continuous_odds_ratios["coefficient"]
)

continuous_odds_ratios[
    "odds_change_percent"
] = (
    continuous_odds_ratios["odds_ratio"] - 1
) * 100


plot_data = (
    continuous_odds_ratios
    .sort_values("odds_change_percent")
    .reset_index(drop=True)
)


continuous_colors = np.where(
    plot_data["odds_change_percent"] >= 0,
    QUALIFYZE_COLORS["higher_odds"],
    QUALIFYZE_COLORS["lower_odds"],
)


fig, ax = plt.subplots(
    figsize=(16, 6),
    constrained_layout=True,
)

fig.patch.set_facecolor(
    QUALIFYZE_COLORS["background"]
)

ax.set_facecolor(
    QUALIFYZE_COLORS["background"]
)


bars = ax.bar(
    x=plot_data["variable"],
    height=plot_data["odds_change_percent"],
    color=continuous_colors,
    alpha=0.9,
    width=0.7,
    zorder=2,
)


# Zero means no estimated change in odds.
ax.axhline(
    y=0,
    color=QUALIFYZE_COLORS["baseline"],
    linewidth=1.3,
    linestyle="--",
    zorder=3,
)


ax.yaxis.set_major_formatter(
    PercentFormatter(
        xmax=100,
        decimals=1,
    )
)


# ax.set_xlabel("")

ax.set_ylabel(
    "Change in VAI/OAI odds for a one-unit increase"
)

ax.set_title(
    "Estimated effect of non-compliance related variables on VAI/OAI odds"
)


# Rotate the longer feature names.
ax.tick_params(
    axis="x",
    labelrotation=55,
    labelsize=12,
    colors=QUALIFYZE_COLORS["text"],
)

ax.tick_params(
    axis="y",
    colors=QUALIFYZE_COLORS["text"],
)


ax.xaxis.label.set_color(
    QUALIFYZE_COLORS["text"]
)

ax.yaxis.label.set_color(
    QUALIFYZE_COLORS["text"]
)

ax.title.set_color(
    QUALIFYZE_COLORS["text"]
)


ax.set_axisbelow(True)

ax.grid(
    axis="y",
    color=QUALIFYZE_COLORS["grid"],
    linewidth=0.8,
    alpha=0.8,
)


# Expand the y-axis so annotations remain visible.
values = plot_data[
    "odds_change_percent"
].to_numpy()

data_min = min(
    0.0,
    float(values.min()),
)

data_max = max(
    0.0,
    float(values.max()),
)

data_range = max(
    data_max - data_min,
    1.0,
)

padding = data_range * 0.20

ax.set_ylim(
    data_min - padding,
    data_max + padding,
)


# Add percentage labels.
for bar, value in zip(
    bars,
    values,
):
    if value >= 0:
        vertical_alignment = "bottom"
        label_offset = 4
    else:
        vertical_alignment = "top"
        label_offset = -4

    ax.annotate(
        text=f"{value:+.2f}%",
        xy=(
            bar.get_x()
            + bar.get_width() / 2,
            value,
        ),
        xytext=(
            0,
            label_offset,
        ),
        textcoords="offset points",
        ha="center",
        va=vertical_alignment,
        fontsize=9,
        color=QUALIFYZE_COLORS["text"],
        clip_on=True,
    )


plt.show()

## 9. Export the complete interpretation table

In [ ]:
export_path = artifact_path.parent / "coefficient_odds_ratios.csv"

coefficient_table.drop(
    columns="absolute_coefficient"
).to_csv(
    export_path,
    index=False,
)

print(f"Saved coefficient interpretation to: {export_path}")

## Interpretation guide

- An odds ratio of **1.25** means 25% higher estimated odds of VAI/OAI, holding the other model inputs constant.
- An odds ratio of **0.80** means 20% lower estimated odds.
- Odds are not the same as probability. Do not describe a 25% increase in odds as a 25-percentage-point increase in probability.
- Continuous variables processed with `StandardScaler` are interpreted per one-standard-deviation increase, not per raw unit.
- Binary variables are interpreted as changing from 0 to 1.
- State indicators are interpreted relative to the dropped reference state.
- L2-regularized coefficients are intentionally shrunk toward zero and should be presented as stable predictive associations rather than causal effects or classical hypothesis tests.